# GPT-v1 — Decoder-only LM (paper-inspired)

Based on: **Radford et al., Improving Language Understanding by Generative Pre-Training (GPT-1, 2018)**

## Paper architecture (reference)
| Piece | Paper GPT-1 |
|---|---|
| Type | Decoder-only Transformer |
| Layers `L` | 12 |
| Hidden `d_model` | 768 |
| Heads | 12 |
| FFN `d_ff` | 3072 (= 4 × d_model) |
| Context | 512 |
| Activation | GELU |
| Positions | Learned embeddings |
| Objective | Next-token LM |
| Optimizer | Adam, LR 2.5e-4, warmup 2000, cosine decay |

## This notebook
1. One **config cell** controls datasets / epochs / layers / batch (no hunting through code)
2. Imports `transformer.py` (source of truth — keep `.py` and this notebook behavior the same)
3. Runs full train → eval → zero-shot samples → optional Gradio chat

`local` mode uses a **smaller** model so a 4GB laptop GPU can train, but the **structure is the same as the paper**.

In [ ]:
# Cell 1 — mode + HF token
import os

# local  = laptop-safe (smaller L/d/heads/ctx)
# full   = paper-like size on a stronger GPU
# kaggle = paper-like size for Kaggle 2xT4
os.environ["RUN_MODE"] = "local"

# Optional: set your HF token for stable dataset downloads
# os.environ["HF_TOKEN"] = "hf_xxx"

print("RUN_MODE =", os.environ["RUN_MODE"])

In [ ]:
# Cell 2 — import transformer.py (source of truth)
import importlib.util
from pathlib import Path

mod_path = Path("transformer.py").resolve()
spec = importlib.util.spec_from_file_location("gptv1", mod_path)
gptv1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gptv1)

print("Loaded:", mod_path)
print("Module RUN_MODE:", gptv1.RUN_MODE)
print("Paper GPT-1 reference:", gptv1.PAPER_GPT1)

In [ ]:
# Cell 3 — SINGLE CONFIG CELL (edit everything here)
# Leave a key out to keep the profile default from transformer.py

CONFIG_OVERRIDE = {
    # ----- schedule -----
    "num_epochs": 5,
    "batch_size": 8,
    "accum_iter": 8,
    "base_lr": 2.5e-4,      # paper max LR
    "warmup": 2000,         # paper warmup steps
    "weight_decay": 0.01,   # paper
    "grad_clip": 1.0,

    # ----- architecture (local-safe GPT-1 family) -----
    # Paper: n_layer=12, n_embd=768, n_head=12, d_ff=3072, max_seq_len=512
    "n_layer": 8,
    "n_embd": 512,
    "n_head": 8,
    "d_ff": 2048,           # 4 * n_embd
    "dropout": 0.1,
    "max_seq_len": 128,
    "block_stride": 96,

    # ----- HF datasets -----
    "train_sources": ["dailydialog", "alpaca", "gsm8k"],
    "valid_sources": ["dailydialog", "gsm8k"],
    "train_pairs": None,          # None = no extra global cap
    "valid_pairs": None,
    "alpaca_train_pairs": 52000,
    "gsm8k_train_pairs": 7000,
    "gsm8k_valid_pairs": 1000,

    # ----- files -----
    "vocab_path": "gptv1_vocab_local.json",
    "file_prefix": "gptv1_local_",
    "save_every_steps": 1200,
}

print("CONFIG_OVERRIDE ready. Keys:", sorted(CONFIG_OVERRIDE.keys()))

In [ ]:
# Cell 4 — preview merged config + architecture banner
cfg = gptv1.get_run_config(CONFIG_OVERRIDE)
print("Merged config:")
for k in sorted(cfg.keys()):
    print(f"  {k}: {cfg[k]}")
print()
gptv1.print_architecture(cfg)

In [ ]:
# Cell 5 — run full pipeline (data → vocab → train/resume → eval → samples)
model, tokenizer, cfg = gptv1.run_pipeline(CONFIG_OVERRIDE)
print("Pipeline finished.")
print("Params:", f"{gptv1.count_parameters(model):,}")

In [ ]:
# Cell 6 — extra zero-shot checks
samples = [
    "hello",
    "solve 23*17",
    "explain recursion in simple words",
    "write a short apology message",
]
for q in samples:
    print("User:", q)
    print("GPT-v1:", gptv1.generate_reply(model, tokenizer, q, max_new_tokens=90))
    print("-" * 60)

In [ ]:
# Cell 7 — optional Gradio chat UI
gptv1.launch_gradio_chat(model, tokenizer)